# Faculty question 9: let GraphCodeBERT make the decision itself

**Question:** why add a Random Forest? Why not tweak the Transformer's last layer and let it decide?

**Test:** give ONE neural network exactly what our hybrid gets (the code + the counts + the ratios) and train it end to end, so its own last layer decides CPU or GPU. Then compare on the same exam (new applications) and the same piles:

| | AMD | NVIDIA |
|---|---|---|
| GraphCodeBERT fine-tuned, numbers as text (done) | 72.5 | 64.5 |
| Hybrid: GraphCodeBERT + Random Forest (done) | 75.8 | 72.1 |
| **This notebook: GraphCodeBERT + numbers, end to end** | ? | ? |

**How to run:**
1. **Runtime → Change runtime type → T4 GPU**, save.
2. **Runtime → Run all.** Allow Google Drive (tick **Select all**).
3. Each run prints its progress. Seed 1 for AMD and NVIDIA comes first (that is the important part); seeds 2 and 3 are repeats.
4. **If you must stop** (or Colab disconnects): every finished pile is saved in `devmap_results` on your Drive. Run all again later and it carries on.
5. Send me a screenshot of the last cell when you stop.

In [ ]:
import os, shutil, subprocess, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - use Runtime > Change runtime type > T4 GPU")
if not os.path.exists("GM_SOURCE_CODE"):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/marco-siino/GM_SOURCE_CODE.git"], check=True)
os.makedirs("data", exist_ok=True)
shutil.copy("GM_SOURCE_CODE/AMD_DS/dataset-devmap-amd.csv", "data")
shutil.copy("GM_SOURCE_CODE/NVIDIA_DS/dataset-devmap-nvidia.csv", "data")
print("Ready.")

## Helper code (no need to read or change)

In [ ]:
%%writefile devmap.py
"""
Shared helpers: load the DevMap dataset and build the four "exams".

Words used here (same as in our chats):
  row      = one line of the dataset (code + two job numbers + answer)
  program  = one piece of OpenCL code (a "kernel"); several rows can share it
  study    = the rows a model learns from   (training set)
  exam     = the rows it is scored on       (test set)

  application = the benchmark a program was taken from (e.g. "npb-3.3-BT");
                one application contains several programs. 71 in total.
  collection  = the benchmark suite an application belongs to (NPB, Rodinia,
                ...). 7 in total.

The four exams, from easiest to hardest:
  old             - the base paper's exam: shuffle the ROWS, cut into 5 piles.
                    Rows of the same program can land on both sides.
  unseen_program  - shuffle the PROGRAMS, cut into 5 piles. Every row of a
                    program is on one side only.
  unseen_app      - shuffle the APPLICATIONS, cut into 5 piles. Needed because
                    programs of one application share the same job numbers,
                    so those numbers give the application away.
  unseen_suite    - hold out one whole collection at a time, 7 rounds.
"""
import os
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "data")
FILES = {"amd": "dataset-devmap-amd.csv", "nvidia": "dataset-devmap-nvidia.csv"}

# The hand-counted features that ship with the dataset (Grewe et al.):
#   comp      = number of compute operations in the code
#   rational  = number of relational (comparison) operations
#   mem       = number of memory accesses
#   localmem  = number of accesses to local memory
#   coalesced = number of memory accesses that are neatly side by side
#   atomic    = number of atomic operations (always 0 here)
#   transfer  = how much data is sent to the device  (job number 1)
#   wgsize    = how the work is batched               (job number 2)
STATIC_FEATURES = ["comp", "rational", "mem", "localmem", "coalesced", "atomic"]
JOB_FEATURES = ["transfer", "wgsize"]

SUITES = ["amd-sdk", "npb", "nvidia-sdk", "parboil", "polybench", "rodinia", "shoc"]


def suite_of(benchmark_name):
    """Which collection a program comes from, read off its name."""
    b = benchmark_name.lower()
    if b.startswith("amd"):
        return "amd-sdk"
    if b.startswith("npb"):
        return "npb"
    if b.startswith("nvidia"):
        return "nvidia-sdk"
    if b.startswith("parboil"):
        return "parboil"
    if b.startswith("polybench"):
        return "polybench"
    if b.startswith("rodinia"):
        return "rodinia"
    if b.startswith("shoc"):
        return "shoc"
    raise ValueError("unknown collection for " + benchmark_name)


def app_of(benchmark_name):
    """'npb-3.3-BT-exact_rhs1' -> 'npb-3.3-BT' (collection, version, application)."""
    m = re.match(r"^(.*?-\d+(?:\.\d+)*)-([^-]+)-(.*)$", benchmark_name)
    if not m:
        raise ValueError("cannot read application from " + benchmark_name)
    return m.group(1) + "-" + m.group(2)


def code_key(src):
    """Two rows are 'the same program' when their code text is identical
    once extra spaces and line breaks are ignored."""
    return re.sub(r"\s+", " ", str(src)).strip()


def load(machine, data_dir=DATA_DIR):
    """Load one answer sheet ('amd' or 'nvidia') with a few helper columns."""
    df = pd.read_csv(os.path.join(data_dir, FILES[machine]), sep=",")
    df = df.loc[:, [c for c in df.columns if not c.startswith("Unnamed")]].copy()
    df["row_id"] = np.arange(len(df))           # stable id = original file order
    df["suite"] = df["benchmark"].map(suite_of)
    df["app"] = df["benchmark"].map(app_of)
    keys = df["src"].map(code_key)
    df["program_id"] = pd.factorize(keys)[0]     # same code text -> same id
    df["label"] = (df["oracle"] == "GPU").astype(int)   # 0 = CPU, 1 = GPU
    return df


# ----------------------------------------------------------------------------
# The four exams. Each returns a list of rounds; a round is a dict with
# 'name', 'study' (row_ids) and 'exam' (row_ids).
# ----------------------------------------------------------------------------
def old_exam(df):
    """Exactly what the base paper's notebooks do:
    df.sample(frac=1, random_state=1) and then KFold(5, shuffle=False)."""
    shuffled = df.sample(frac=1, random_state=1).reset_index(drop=True)
    rounds = []
    for i, (tr, te) in enumerate(KFold(n_splits=5, shuffle=False).split(shuffled)):
        rounds.append({
            "name": f"pile{i + 1}",
            "study": shuffled.iloc[tr]["row_id"].tolist(),
            "exam": shuffled.iloc[te]["row_id"].tolist(),
        })
    return rounds


def group_piles(groups, labels, n_piles=5, seed=1):
    """Deal whole groups (programs or applications) into n_piles piles.

    Our own version instead of scikit-learn's StratifiedGroupKFold, because
    that one gives different piles in different library versions (Colab's
    1.6 put 364 rows in one AMD pile and 15 in another). This uses only
    numpy's fixed random generator, so the piles are the same everywhere.

    How it deals: shuffle the groups (seed), then take them biggest first and
    put each one in the pile where it keeps the piles most even, both in
    size and in their share of GPU answers. Returns one pile number per row."""
    groups = np.asarray(groups)
    labels = np.asarray(labels)
    names, inverse = np.unique(groups, return_inverse=True)
    sizes = np.bincount(inverse)
    gpus = np.bincount(inverse, weights=labels)
    order = np.random.RandomState(seed).permutation(len(names))
    order = order[np.argsort(-sizes[order], kind="stable")]       # biggest first, ties in shuffled order
    target_rows = len(groups) / n_piles
    target_gpu = max(labels.sum() / n_piles, 1.0)
    pile_rows = np.zeros(n_piles)
    pile_gpu = np.zeros(n_piles)
    pile_of_group = np.empty(len(names), dtype=int)
    for g in order:
        costs = []
        for p in range(n_piles):
            rows, gpu = pile_rows.copy(), pile_gpu.copy()
            rows[p] += sizes[g]
            gpu[p] += gpus[g]
            costs.append((((rows - target_rows) / target_rows) ** 2).sum()
                         + (((gpu - target_gpu) / target_gpu) ** 2).sum())
        best = int(np.argmin(costs))                               # ties -> lowest pile number
        pile_of_group[g] = best
        pile_rows[best] += sizes[g]
        pile_gpu[best] += gpus[g]
    return pile_of_group[inverse]


def fair_exam_unseen_programs(df, seed=1):
    """5 rounds; all rows of a program stay on the same side. Piles are also
    kept roughly balanced in size and CPU/GPU answers."""
    pile = group_piles(df["program_id"], df["label"], 5, seed)
    rounds = []
    for i in range(5):
        tr, te = np.where(pile != i)[0], np.where(pile == i)[0]
        rounds.append({
            "name": f"pile{i + 1}",
            "study": df.iloc[tr]["row_id"].tolist(),
            "exam": df.iloc[te]["row_id"].tolist(),
        })
    return rounds


def fair_exam_unseen_apps(df, seed=1):
    """5 rounds; all programs of an application stay on the same side. If the
    exact same code also exists under another application, those study rows
    are dropped so the exam code is truly unseen."""
    pile = group_piles(df["app"], df["label"], 5, seed)
    rounds = []
    for i in range(5):
        tr, te = np.where(pile != i)[0], np.where(pile == i)[0]
        exam = df.iloc[te]
        study = df.iloc[tr]
        study = study[~study["program_id"].isin(exam["program_id"])]
        rounds.append({
            "name": f"pile{i + 1}",
            "study": study["row_id"].tolist(),
            "exam": exam["row_id"].tolist(),
        })
    return rounds


def fair_exam_unseen_collection(df):
    """7 rounds; each round holds out one whole collection. If a held-out
    program's exact code also exists in another collection, those study rows
    are dropped so the exam code is truly unseen."""
    rounds = []
    for suite in SUITES:
        exam = df[df["suite"] == suite]
        study = df[(df["suite"] != suite) & (~df["program_id"].isin(exam["program_id"]))]
        rounds.append({
            "name": suite,
            "study": study["row_id"].tolist(),
            "exam": exam["row_id"].tolist(),
        })
    return rounds


def build_exam(df, exam, seed=1):
    if exam == "old":
        return old_exam(df)
    if exam == "unseen_program":
        return fair_exam_unseen_programs(df, seed)
    if exam == "unseen_app":
        return fair_exam_unseen_apps(df, seed)
    if exam == "unseen_suite":
        return fair_exam_unseen_collection(df)
    raise ValueError(exam)


EXAMS = ["old", "unseen_program", "unseen_app", "unseen_suite"]
RESHUFFLED_EXAMS = ["unseen_program", "unseen_app"]   # these change with the shuffle seed


def leak_report(df, rounds):
    """For each round: how many exam rows have code that is also in the
    study rows (seen) vs. not (new)."""
    pid = df.set_index("row_id")["program_id"]
    out = []
    for r in rounds:
        study_programs = set(pid.loc[r["study"]])
        seen = sum(1 for x in r["exam"] if pid.loc[x] in study_programs)
        out.append({"round": r["name"], "study_rows": len(r["study"]),
                    "exam_rows": len(r["exam"]), "exam_seen_code": seen,
                    "exam_new_code": len(r["exam"]) - seen})
    return pd.DataFrame(out)

In [ ]:
%%writefile features.py
"""
Hand-made features shared by the name-tag test and the hybrid model.

  counts  - the six code counts that ship with DevMap (comp, rational, mem,
            localmem, coalesced, atomic)
  raw job numbers - transfer and wgsize exactly as stored. Nearly unique per
            application, so they can act as a "name tag".
  ratios  - scale-free versions that keep the useful information (how much
            data per operation, how much work per work-item) without being a
            name tag on their own:
              transfer_per_op = transfer / (comp + mem)   data moved per operation
              coalesced_share = coalesced / mem           share of tidy memory accesses
              local_share     = localmem / mem            share of local-memory accesses
              comp_per_mem    = comp / mem                arithmetic intensity
              ops_per_item    = (comp + mem) / wgsize     work per work-item in a batch
  grewe4  - the four features Grewe et al. (2013) designed. F3 still multiplies
            by the raw wgsize.
"""
import numpy as np
import pandas as pd

import devmap


def _safe(a, b):
    return (a / b.replace(0, np.nan)).fillna(0.0)


def counts(df):
    return df[devmap.STATIC_FEATURES].astype(float)


def job_numbers(df):
    return df[devmap.JOB_FEATURES].astype(float)


def ratios(df):
    X = df[devmap.STATIC_FEATURES + devmap.JOB_FEATURES].astype(float)
    ops = X["comp"] + X["mem"]
    return pd.DataFrame({
        "transfer_per_op": _safe(X["transfer"], ops),
        "coalesced_share": _safe(X["coalesced"], X["mem"]),
        "local_share": _safe(X["localmem"], X["mem"]),
        "comp_per_mem": _safe(X["comp"], X["mem"]),
        "ops_per_item": _safe(ops, X["wgsize"]),
    }, index=df.index)


def grewe4(df):
    X = df[devmap.STATIC_FEATURES + devmap.JOB_FEATURES].astype(float)
    r = ratios(df)
    return pd.DataFrame({
        "F1": r["transfer_per_op"], "F2": r["coalesced_share"],
        "F3": r["local_share"] * X["wgsize"], "F4": r["comp_per_mem"],
    }, index=df.index)


def forest(df):
    """Our original Random Forest baseline: counts + raw job numbers + Grewe's four."""
    return pd.concat([counts(df), job_numbers(df), grewe4(df)], axis=1)

In [ ]:
%%writefile train_lib.py
"""
Re-creates the base paper's Transformer training, step for step, and lets the
same model sit any of our four exams.

Copied from the authors' public notebooks (github.com/marco-siino/GM_SOURCE_CODE):
  * input text  = "<transfer> - <wgsize> - <code>", extra spaces removed
  * at most 512 tokens, padded to 512
  * model       = pretrained encoder -> Linear(768,768) -> ReLU -> Dropout(0.3)
                  -> Linear(768,2), reading the first token's output
  * training    = Adam, learning rate 1e-05, batch of 4, 10 passes (epochs),
                  scored once after the last pass
What we add: a fixed random seed per run (theirs was not fixed), per-row
predictions saved to disk, and the choice of exam.
"""
import glob
import os
import random
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import devmap

# Short name -> pretrained model on the Hugging Face hub.
MODELS = {
    "codeberta": "huggingface/CodeBERTa-language-id",                            # paper's best
    "distilbert": "distilbert/distilbert-base-uncased-finetuned-sst-2-english",  # as in their notebook
    "codebert": "microsoft/codebert-base",                                       # their notebook loads a local copy
    "graphcodebert": "microsoft/graphcodebert-base",                             # OURS - not in the paper
}

# The paper's own scores on the old exam (Tables 3 and 4), per pile and average.
PAPER_OLD_EXAM = {
    ("codeberta", "amd"): [0.956, 0.897, 0.956, 0.890, 0.941],
    ("codeberta", "nvidia"): [0.934, 0.868, 0.904, 0.875, 0.875],
    ("distilbert", "amd"): [0.934, 0.897, 0.971, 0.875, 0.963],
    ("distilbert", "nvidia"): [0.890, 0.868, 0.875, 0.824, 0.838],
    ("codebert", "amd"): [0.956, 0.890, 0.956, 0.897, 0.897],
    ("codebert", "nvidia"): [0.912, 0.824, 0.882, 0.853, 0.860],
}

DEFAULT_SETTINGS = {
    "max_len": 512,
    "train_batch": 4,
    "eval_batch": 16,      # only affects speed, not the answers
    "epochs": 10,
    "lr": 1e-05,
    "dropout": 0.3,
    "fast": False,         # True = our speed-ups (see train_and_predict_fast); False = exactly the paper's way
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_text(df):
    """The paper's input format: the two job numbers, then the code."""
    text = df["transfer"].astype(str) + " - " + df["wgsize"].astype(str) + " - " + df["src"].astype(str)
    return [" ".join(t.split()) for t in text]


def encode(tokenizer, texts, max_len):
    enc = tokenizer(list(texts), padding="max_length", truncation=True,
                    max_length=max_len, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]


class Classifier(nn.Module):
    """Same head as the paper's CodeBERTaClass."""

    def __init__(self, encoder, dropout=0.3):
        super().__init__()
        self.l1 = encoder
        hidden = encoder.config.hidden_size
        self.pre_classifier = nn.Linear(hidden, hidden)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 2)

    def forward(self, input_ids, attention_mask):
        hidden_state = self.l1(input_ids=input_ids, attention_mask=attention_mask)[0]
        pooler = hidden_state[:, 0]
        pooler = torch.relu(self.pre_classifier(pooler))
        pooler = self.dropout(pooler)
        return self.classifier(pooler)


def _pad(seqs, pad_id):
    """Pad a handful of token lists only up to the longest one among them."""
    longest = max(len(x) for x in seqs)
    ids = torch.full((len(seqs), longest), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), longest), dtype=torch.long)
    for i, x in enumerate(seqs):
        ids[i, :len(x)] = torch.tensor(x, dtype=torch.long)
        mask[i, :len(x)] = 1
    return ids, mask


def train_and_predict_fast(study, exam, tokenizer, make_encoder, device, settings, seed, log=print, text_fn=None):
    """Same model, same settings, same number of passes as the paper's way, with
    two speed-ups that do not change what the model is asked to learn:
      1. each batch is padded only to its own longest input, not always to 512
         tokens, so less work is spent on empty filler;
      2. on a GPU, the arithmetic runs in half precision (16-bit numbers)."""
    set_seed(seed)
    text_fn = text_fn or make_text
    pad_id = tokenizer.pad_token_id if getattr(tokenizer, "pad_token_id", None) is not None else 0
    tr_seqs = tokenizer(text_fn(study), truncation=True, max_length=settings["max_len"])["input_ids"]
    ex_seqs = tokenizer(text_fn(exam), truncation=True, max_length=settings["max_len"])["input_ids"]
    y_tr = study["label"].to_numpy()

    use_amp = str(device).startswith("cuda")
    model = Classifier(make_encoder(), settings["dropout"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=settings["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    loss_fn = nn.CrossEntropyLoss()
    order_gen = torch.Generator().manual_seed(seed)
    bs = settings["train_batch"]

    for epoch in range(settings["epochs"]):
        model.train()
        total_loss, correct, seen = 0.0, 0, 0
        t0 = time.time()
        order = torch.randperm(len(tr_seqs), generator=order_gen).tolist()
        for start in range(0, len(order), bs):
            idx = order[start:start + bs]
            ids, mask = _pad([tr_seqs[i] for i in idx], pad_id)
            y = torch.tensor(y_tr[idx], dtype=torch.long)
            ids, mask, y = ids.to(device), mask.to(device), y.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                out = model(ids, mask)
                loss = loss_fn(out.float(), y)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * y.size(0)
            correct += (out.argmax(dim=1) == y).sum().item()
            seen += y.size(0)
        log(f"      pass {epoch + 1:2d}/{settings['epochs']}  loss {total_loss / seen:.4f}  "
            f"study accuracy {100 * correct / seen:5.1f}%  ({time.time() - t0:.0f}s)")

    model.eval()
    by_length = sorted(range(len(ex_seqs)), key=lambda i: len(ex_seqs[i]))   # similar lengths together
    prob_gpu = np.zeros(len(ex_seqs))
    with torch.no_grad():
        for start in range(0, len(by_length), settings["eval_batch"]):
            idx = by_length[start:start + settings["eval_batch"]]
            ids, mask = _pad([ex_seqs[i] for i in idx], pad_id)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                out = model(ids.to(device), mask.to(device))
            prob_gpu[idx] = torch.softmax(out.float(), dim=1)[:, 1].cpu().numpy()
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return (prob_gpu >= 0.5).astype(int), prob_gpu


def train_and_predict(study, exam, tokenizer, make_encoder, device, settings, seed, log=print):
    """Train on the study rows, then answer the exam rows.
    Returns (predicted labels, probability of GPU) in exam-row order."""
    if settings.get("fast"):
        return train_and_predict_fast(study, exam, tokenizer, make_encoder, device, settings, seed, log)
    set_seed(seed)
    ids_tr, mask_tr = encode(tokenizer, make_text(study), settings["max_len"])
    y_tr = torch.tensor(study["label"].to_numpy(), dtype=torch.long)
    ids_ex, mask_ex = encode(tokenizer, make_text(exam), settings["max_len"])

    model = Classifier(make_encoder(), settings["dropout"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=settings["lr"])
    loss_fn = nn.CrossEntropyLoss()
    shuffle_gen = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(ids_tr, mask_tr, y_tr), batch_size=settings["train_batch"],
                        shuffle=True, generator=shuffle_gen)

    for epoch in range(settings["epochs"]):
        model.train()
        total_loss, correct, seen = 0.0, 0, 0
        t0 = time.time()
        for ids, mask, y in loader:
            ids, mask, y = ids.to(device), mask.to(device), y.to(device)
            out = model(ids, mask)
            loss = loss_fn(out, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * y.size(0)
            correct += (out.argmax(dim=1) == y).sum().item()
            seen += y.size(0)
        log(f"      pass {epoch + 1:2d}/{settings['epochs']}  loss {total_loss / seen:.4f}  "
            f"study accuracy {100 * correct / seen:5.1f}%  ({time.time() - t0:.0f}s)")

    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(ids_ex), settings["eval_batch"]):
            out = model(ids_ex[i:i + settings["eval_batch"]].to(device),
                        mask_ex[i:i + settings["eval_batch"]].to(device))
            probs.append(torch.softmax(out.float(), dim=1)[:, 1].cpu())
    prob_gpu = torch.cat(probs).numpy()
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return (prob_gpu >= 0.5).astype(int), prob_gpu


def result_file(out_dir, model_key, machine, exam, split_seed, train_seed):
    return os.path.join(out_dir, f"preds__{model_key}__{machine}__{exam}__split{split_seed}__train{train_seed}.csv")


def run_experiment(model_key, machine, exam, data_dir, out_dir, split_seed=1, train_seed=1,
                   settings=None, tokenizer=None, make_encoder=None, device=None, log=print,
                   trainer=None, label=None):
    """One model, one machine, one exam, all rounds. Saves after every round,
    and if the file already has some rounds (e.g. Colab disconnected) it
    carries on from where it stopped. `trainer` (default train_and_predict) and
    `label` (default "<model_key>[+fast]") let other experiments reuse this loop."""
    trainer = trainer or train_and_predict
    settings = {**DEFAULT_SETTINGS, **(settings or {})}
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(out_dir, exist_ok=True)

    if tokenizer is None or make_encoder is None:
        from transformers import AutoModel, AutoTokenizer
        hub_name = MODELS[model_key]
        tokenizer = AutoTokenizer.from_pretrained(hub_name)
        make_encoder = lambda hub_name=hub_name: AutoModel.from_pretrained(hub_name)   # name fixed now

    label = label or model_key + ("+fast" if settings.get("fast") else "")   # e.g. "codeberta+fast"
    df = devmap.load(machine, data_dir).set_index("row_id", drop=False)
    rounds = devmap.build_exam(df, exam, seed=split_seed)
    path = result_file(out_dir, label, machine, exam, split_seed, train_seed)
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()

    log(f"\n=== {label} | {machine} | exam '{exam}' | shuffle seed {split_seed} | training seed {train_seed} ===")
    # A saved round only counts if it holds exactly this round's exam rows. Rounds saved
    # with different piles (e.g. an older version of the splitting code) are redone.
    done_rounds = set()
    if len(done):
        wanted = {r["name"]: set(r["exam"]) for r in rounds}
        keep = []
        for round_name, part in done.groupby("round", sort=False):
            want = wanted.get(round_name)
            if want is not None and set(part["row_id"]) == want and len(part) == len(want):
                done_rounds.add(round_name)
                keep.append(part)
            else:
                log(f"   round {round_name}: saved with different exam rows, will redo it")
        if keep:
            done = pd.concat(keep, ignore_index=True)
            done.to_csv(path, index=False)
        else:
            done = pd.DataFrame()
            os.remove(path)
    for r in rounds:
        if r["name"] in done_rounds:
            log(f"   round {r['name']}: already done, skipping")
            continue
        study, ex = df.loc[r["study"]], df.loc[r["exam"]]
        log(f"   round {r['name']}: study {len(study)} rows, exam {len(ex)} rows")
        pred, prob = trainer(study, ex, tokenizer, make_encoder, device, settings, train_seed, log)
        part = pd.DataFrame({
            "model": label, "machine": machine, "exam": exam,
            "split_seed": split_seed, "train_seed": train_seed, "round": r["name"],
            "row_id": ex["row_id"].to_numpy(), "benchmark": ex["benchmark"].to_numpy(),
            "suite": ex["suite"].to_numpy(), "app": ex["app"].to_numpy(),
            "program_id": ex["program_id"].to_numpy(),
            "code_seen_in_study": ex["program_id"].isin(set(study["program_id"])).to_numpy(),
            "transfer": ex["transfer"].to_numpy(), "wgsize": ex["wgsize"].to_numpy(),
            "label": ex["label"].to_numpy(), "pred": pred, "prob_gpu": prob,
        })
        done = pd.concat([done, part], ignore_index=True)
        done.to_csv(path, index=False)
        log(f"   round {r['name']}: exam accuracy {100 * (part['label'] == part['pred']).mean():.1f}%")
    log(f"   overall accuracy {100 * (done['label'] == done['pred']).mean():.1f}%  -> saved {os.path.basename(path)}")
    return done


def load_predictions(out_dir):
    files = sorted(glob.glob(os.path.join(out_dir, "preds__*.csv")))
    if not files:
        return pd.DataFrame()
    preds = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    preds["correct"] = (preds["label"] == preds["pred"]).astype(int)
    preds["run"] = "split" + preds["split_seed"].astype(str) + "_train" + preds["train_seed"].astype(str)
    return preds


def summarise(out_dir):
    """Tables of everything finished so far. Returns a dict of DataFrames."""
    preds = load_predictions(out_dir)
    if preds.empty:
        return {}
    exam_order = {e: i for i, e in enumerate(devmap.EXAMS)}

    per_run = preds.groupby(["model", "machine", "exam", "run"])["correct"].mean().reset_index()
    main = per_run.groupby(["model", "machine", "exam"])["correct"].agg(
        accuracy="mean", spread="std", runs="count").reset_index()
    main["accuracy"] = (100 * main["accuracy"]).round(1)
    main["spread"] = (100 * main["spread"].fillna(0)).round(1)
    base = lambda m: m.split("+")[0]
    main["paper"] = [
        round(100 * float(np.mean(PAPER_OLD_EXAM[(base(m), k)])), 1)
        if e == "old" and (base(m), k) in PAPER_OLD_EXAM else np.nan
        for m, k, e in zip(main["model"], main["machine"], main["exam"])]
    main = main.sort_values(["model", "machine", "exam"], key=lambda c: c.map(exam_order) if c.name == "exam" else c)

    old = preds[preds["exam"] == "old"]
    piles = seen_new = pd.DataFrame()
    if len(old):
        piles = (100 * old.groupby(["model", "machine", "round"])["correct"].mean()).round(1).unstack()
        piles["average"] = piles.mean(axis=1).round(1)
        present = {(m.split("+")[0], k) for m, k in piles.index}
        paper_rows = {(m, k): [round(100 * v, 1) for v in vals] + [round(100 * float(np.mean(vals)), 1)]
                      for (m, k), vals in PAPER_OLD_EXAM.items() if (m, k) in present}
        for (m, k), vals in paper_rows.items():
            piles.loc[(m + " (paper)", k), :] = vals
        piles = piles.sort_index()
        seen_new = (100 * old.groupby(["model", "machine", "code_seen_in_study"])["correct"].mean()).round(1).unstack()
        seen_new = seen_new.rename(columns={False: "new code", True: "seen code"})

    suite = preds[preds["exam"] == "unseen_suite"]
    per_suite = pd.DataFrame()
    if len(suite):
        per_suite = (100 * suite.groupby(["model", "machine", "suite"])["correct"].mean()).round(1).unstack()

    return {"main": main, "old_exam_piles": piles, "old_exam_seen_vs_new": seen_new, "per_collection": per_suite}

In [ ]:
%%writefile 10_end_to_end.py
"""
Faculty question 9: "Why not let the Transformer make the decision itself
(tweak its last layer) instead of adding a Random Forest?"

Our hybrid = frozen GraphCodeBERT reading the code + counts + ratios, decided
by a Random Forest. Here the SAME inputs go into ONE neural network, trained
end to end, and its own last layer makes the decision:

    code text -> GraphCodeBERT -> first-token vector -> Linear -> ReLU --+
                                                                        |-> join -> Dropout -> Linear -> CPU/GPU
    counts + ratios (log, standardised) -> Linear(., 32) -> ReLU -------+

  * The text is the code only. The numbers go in as numbers, not as text.
  * Same training recipe as our fine-tuned GraphCodeBERT (Adam, 10 passes,
    batch 4, learning rate 1e-5, fast mode). The two new number layers start
    from scratch, so they get a larger learning rate (1e-3); everything else
    keeps 1e-5.
  * Same exam (unseen_app, shuffle seed 1) and same piles, 3 training seeds,
    so it lines up row by row with graphcodebert+fast (72.5 / 64.5) and with
    the hybrid (75.8 / 72.1).
Results are saved as preds__graphcodebert+e2e__<machine>__unseen_app__split1__train<seed>.csv
in the same format as all other fine-tuned runs, and resume after a disconnect.
"""
import time

import numpy as np
import pandas as pd
import torch
from torch import nn

import devmap
import features as F
import train_lib

LABEL = "graphcodebert+e2e"
NEW_LAYER_LR = 1e-3


def number_inputs(df):
    """The hybrid's extra inputs: six code counts + five scale-free ratios."""
    X = pd.concat([F.counts(df), F.ratios(df)], axis=1).astype(float)
    return np.log1p(X.clip(lower=0).to_numpy())


class NumberClassifier(nn.Module):
    def __init__(self, encoder, n_numbers, dropout=0.3, number_width=32):
        super().__init__()
        self.l1 = encoder
        hidden = encoder.config.hidden_size
        self.pre_classifier = nn.Linear(hidden, hidden)
        self.numbers = nn.Sequential(nn.Linear(n_numbers, number_width), nn.ReLU())
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden + number_width, 2)

    def new_layers(self):
        return list(self.numbers.parameters()) + list(self.classifier.parameters())

    def forward(self, input_ids, attention_mask, numbers):
        h = self.l1(input_ids=input_ids, attention_mask=attention_mask)[0][:, 0]
        h = torch.relu(self.pre_classifier(h))
        joined = torch.cat([h, self.numbers(numbers)], dim=1)
        return self.classifier(self.dropout(joined))


def train_and_predict_e2e(study, exam, tokenizer, make_encoder, device, settings, seed, log=print):
    """Like train_lib.train_and_predict_fast, plus the number branch."""
    train_lib.set_seed(seed)
    pad_id = tokenizer.pad_token_id if getattr(tokenizer, "pad_token_id", None) is not None else 0
    code = lambda d: [" ".join(str(s).split()) for s in d["src"]]
    tr_seqs = tokenizer(code(study), truncation=True, max_length=settings["max_len"])["input_ids"]
    ex_seqs = tokenizer(code(exam), truncation=True, max_length=settings["max_len"])["input_ids"]
    n_tr, n_ex = number_inputs(study), number_inputs(exam)
    mu, sd = n_tr.mean(axis=0), n_tr.std(axis=0) + 1e-6          # scaled with study rows only
    n_tr = torch.tensor((n_tr - mu) / sd, dtype=torch.float32)
    n_ex = torch.tensor((n_ex - mu) / sd, dtype=torch.float32)
    y_tr = study["label"].to_numpy()

    use_amp = str(device).startswith("cuda")
    model = NumberClassifier(make_encoder(), n_tr.shape[1], settings["dropout"]).to(device)
    new_ids = {id(p) for p in model.new_layers()}
    old = [p for p in model.parameters() if id(p) not in new_ids]
    optimizer = torch.optim.Adam([{"params": old, "lr": settings["lr"]},
                                  {"params": model.new_layers(), "lr": settings.get("new_layer_lr", NEW_LAYER_LR)}])
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    loss_fn = nn.CrossEntropyLoss()
    order_gen = torch.Generator().manual_seed(seed)
    bs = settings["train_batch"]

    for epoch in range(settings["epochs"]):
        model.train()
        total_loss, correct, seen, t0 = 0.0, 0, 0, time.time()
        order = torch.randperm(len(tr_seqs), generator=order_gen).tolist()
        for start in range(0, len(order), bs):
            idx = order[start:start + bs]
            ids, mask = train_lib._pad([tr_seqs[i] for i in idx], pad_id)
            y = torch.tensor(y_tr[idx], dtype=torch.long).to(device)
            nums = n_tr[idx].to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                out = model(ids.to(device), mask.to(device), nums)
                loss = loss_fn(out.float(), y)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * y.size(0)
            correct += (out.argmax(dim=1) == y).sum().item()
            seen += y.size(0)
        log(f"      pass {epoch + 1:2d}/{settings['epochs']}  loss {total_loss / seen:.4f}  "
            f"study accuracy {100 * correct / seen:5.1f}%  ({time.time() - t0:.0f}s)")

    model.eval()
    by_length = sorted(range(len(ex_seqs)), key=lambda i: len(ex_seqs[i]))
    prob_gpu = np.zeros(len(ex_seqs))
    with torch.no_grad():
        for start in range(0, len(by_length), settings["eval_batch"]):
            idx = by_length[start:start + settings["eval_batch"]]
            ids, mask = train_lib._pad([ex_seqs[i] for i in idx], pad_id)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                out = model(ids.to(device), mask.to(device), n_ex[idx].to(device))
            prob_gpu[idx] = torch.softmax(out.float(), dim=1)[:, 1].cpu().numpy()
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return (prob_gpu >= 0.5).astype(int), prob_gpu


def run(machines=("amd", "nvidia"), seeds=(1, 2, 3), data_dir=devmap.DATA_DIR, out_dir="results",
        settings=None, tokenizer=None, make_encoder=None, device=None, log=print):
    settings = {**train_lib.DEFAULT_SETTINGS, "fast": True, **(settings or {})}
    for seed in seeds:                       # seed 1 for both machines first, then the repeats
        for machine in machines:
            train_lib.run_experiment("graphcodebert", machine, "unseen_app", data_dir, out_dir,
                                     split_seed=1, train_seed=seed, settings=settings,
                                     tokenizer=tokenizer, make_encoder=make_encoder, device=device,
                                     log=log, trainer=train_and_predict_e2e, label=LABEL)


def summary(out_dir, hybrid_csv=None):
    """Accuracy per run, next to fine-tuned GraphCodeBERT if its files are in out_dir."""
    preds = train_lib.load_predictions(out_dir)
    preds = preds[(preds["exam"] == "unseen_app") & (preds["split_seed"] == 1)
                  & preds["model"].isin([LABEL, "graphcodebert+fast", "codeberta+fast"])]
    per_run = (100 * preds.groupby(["model", "machine", "train_seed"])["correct"].mean()).round(1)
    table = per_run.groupby(["model", "machine"]).agg(["mean", "count"]).round(1)
    table.columns = ["accuracy (mean of runs)", "runs"]
    return table.reset_index()

## Settings

In [ ]:
import os
MACHINES = ["amd", "nvidia"]
SEEDS = [1, 2, 3]          # seed 1 first for both machines; 2 and 3 are repeats

RESULTS_DIR = "results"
try:
    from google.colab import drive
    drive.mount("/content/drive")                      # tick "Select all"
    RESULTS_DIR = "/content/drive/MyDrive/devmap_results"
except ImportError:
    pass
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Saving to:", RESULTS_DIR)

## Train and sit the exam (resumable)

In [ ]:
import importlib
E = importlib.import_module("10_end_to_end")
E.run(machines=MACHINES, seeds=SEEDS, data_dir="data", out_dir=RESULTS_DIR)
print("\nAll requested runs are finished.")

## Results so far (send Claude a screenshot)

In [ ]:
import importlib, pandas as pd
E = importlib.import_module("10_end_to_end")
pd.set_option("display.width", 200)
print(E.summary(RESULTS_DIR).to_string(index=False))
print("\nFor comparison, the hybrid (GraphCodeBERT + Random Forest): AMD 75.8, NVIDIA 72.1")

## Download the results to your laptop too (a backup in case Drive fails)

In [ ]:
import glob, os, shutil
from google.colab import files
os.makedirs("backup", exist_ok=True)
for f in glob.glob(os.path.join(RESULTS_DIR, "preds__graphcodebert+e2e__*")):
    shutil.copy(f, "backup")
shutil.make_archive("results_backup", "zip", "backup")
print("Files in backup:", sorted(os.listdir("backup")))
files.download("results_backup.zip")